# Ingestion d'un fichier CSV

In [48]:
# charger les fichiers CSV
import pandas as pd

df = pd.read_csv("../data/csv_excel_files/tickets_support.csv", sep=",")
df.head()

,id_ticket,client,priorite,date_creation,description,agent
0,101,ACME Corp,Haute,2024-02-10,Le client signale un problème d'accès à la pla...,Okan
1,102,Globex,Moyenne,2024-02-11,Erreur intermittente lors du chargement des da...,Sarah
2,103,Stark Industries,Critique,2024-02-12,Crash complet du module de facturation lors de...,Julien
3,104,Wayne Enterprises,Basse,2024-02-13,Demande d'amélioration : ajouter une fonctionn...,Mila


## Convertir une ligne en texte

In [11]:
def row_to_text(row):
    return "\n".join([f"{col}: {row[col]}" for col in row.index]) 

In [12]:
for rows in df.iterrows():
    text = row_to_text(rows[1])
    print(text)

id_ticket: 101
client: ACME Corp
priorite: Haute
date_creation: 2024-02-10
description: Le client signale un problème d'accès à la plateforme après une mise à jour.
agent: Okan
id_ticket: 102
client: Globex
priorite: Moyenne
date_creation: 2024-02-11
description: Erreur intermittente lors du chargement des dashboards internes.
agent: Sarah
id_ticket: 103
client: Stark Industries
priorite: Critique
date_creation: 2024-02-12
description: Crash complet du module de facturation lors de l'import d'un fichier XML.
agent: Julien
id_ticket: 104
client: Wayne Enterprises
priorite: Basse
date_creation: 2024-02-13
description: Demande d'amélioration : ajouter une fonctionnalité d'export PDF pour les rapports.
agent: Mila


## Transformer tout le CSV en Document RAG

In [19]:
from langchain_core.documents import Document
docs = []

for index, row in df.iterrows():
    text = row_to_text(row)
    metadata = {"source": "tickets_support.csv", "row_index": index}
    doc = Document(page_content=text, metadata=metadata)
    docs.append(doc)

print(f"Nombre de documents créés : {len(docs)}")
print("==============================================================")
print(f"Exemple de document : \n {docs[0].page_content}")
print("==============================================================")
print(f"Exemple de métadonnées : \n {docs[0].metadata}")

Nombre de documents créés : 4
Exemple de document : 
 id_ticket: 101
client: ACME Corp
priorite: Haute
date_creation: 2024-02-10
description: Le client signale un problème d'accès à la plateforme après une mise à jour.
agent: Okan
Exemple de métadonnées : 
 {'source': 'tickets_support.csv', 'row_index': 0}


In [24]:
for i, d in enumerate(docs[:2]):
    print("="*50)
    print("DOCUMENT", i)
    print("="*50)
    print(d.page_content, "\n")

DOCUMENT 0
id_ticket: 101
client: ACME Corp
priorite: Haute
date_creation: 2024-02-10
description: Le client signale un problème d'accès à la plateforme après une mise à jour.
agent: Okan 

DOCUMENT 1
id_ticket: 102
client: Globex
priorite: Moyenne
date_creation: 2024-02-11
description: Erreur intermittente lors du chargement des dashboards internes.
agent: Sarah 



# Chunker un document CSV

## Importer le splitter + Application du splitter sur le document CSV

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
)

csv_chunks = splitter.split_documents(docs)

In [30]:
len(csv_chunks)

4

In [31]:
csv_chunks[1].page_content

'id_ticket: 102\nclient: Globex\npriorite: Moyenne\ndate_creation: 2024-02-11\ndescription: Erreur intermittente lors du chargement des dashboards internes.\nagent: Sarah'

# Ingestion Excel

In [53]:
# Lire les feuilles du fichier Excel
import pandas as pd

xls = pd.ExcelFile("../data/csv_excel_files/support_data_advanced.xlsx")

xls.sheet_names

['tickets_support', 'support_stats']

In [54]:
df_xls1 = pd.read_excel(xls, sheet_name=xls.sheet_names[0])
df_xls2 = pd.read_excel(xls, sheet_name=xls.sheet_names[1])

## Convertir uen feuille Excel en document RAG

In [55]:
docs_xls1 = [Document(page_content=row_to_text(row), metadata={"source": "excel", "sheet": "tickets_support", "row_index": index}) for index, row in df_xls1.iterrows()]
docs_xls2 = [Document(page_content=row_to_text(row), metadata={"source": "excel", "sheet": "support_data", "row_index": index}) for index, row in df_xls2.iterrows()]

print(f"Nombre de documents créés à partir de la feuille 1 : {len(docs_xls1)}")
print(f"Nombre de documents créés à partir de la feuille 2 : {len(docs_xls2)}")

print("contenu d'un document de la feuille 1 :")
print(docs_xls1[0].page_content)
print("==============================================================")

print("contenu d'un document de la feuille 2 :")
print(docs_xls2[0].page_content)
print("==============================================================")


Nombre de documents créés à partir de la feuille 1 : 4
Nombre de documents créés à partir de la feuille 2 : 4
contenu d'un document de la feuille 1 :
id_ticket: 201
client: Umbrella Corp
priorite: Critique
date_creation: 2024-03-01
description: Erreur critique sur l'API d'authentification.
agent: Okan
contenu d'un document de la feuille 2 :
mois: Janvier
tickets_crees: 134.0
tickets_resolus: 120.0
satisfaction_client: 4.2


# Nettoyage d'un fichier CSV

## Charger les fichiers CSV

In [56]:
import pandas as pd

df = pd.read_csv("../data/csv_excel_files/support_dirty.csv", sep=",")
df.head()

,ticket_id,client,priority,creation_date,description
0,301.0,Stark Industries,HIGH,2024/01/01,BUG: dashboard cannot load properly
1,302.0,NaN,low,01-02-2024,missing value
2,302.0,Stark Industries,LOW,2024/01/02,Duplicate line from line 2
3,303.0,Oscorp,CRITICAL,NaN,NaN
4,NaN,Wayne Ent.,NaN,BAD_DATE,Strange character line ☃


In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ticket_id      4 non-null      float64
 1   client         4 non-null      str    
 2   priority       4 non-null      str    
 3   creation_date  4 non-null      str    
 4   description    4 non-null      str    
dtypes: float64(1), str(4)
memory usage: 332.0 bytes


In [58]:
df.isna().sum()

ticket_id        1
client           1
priority         1
creation_date    1
description      1
dtype: int64

# Création d'une classe professionnel pour appliquer le nettoyage sur un fichier CSV

In [59]:
class ingest_csv_excel:
    def __init__(self):
        # Standardisation des priorités
        self.priority_map = {
            "high": "Haute",
            "low": "Basse",
            "critical": "Critique",
            "n/a": None,
            "na": None,
        }

    def clean_dates(self, df, col):
        def parse(value):
            try:
                return pd.to_datetime(value, errors="raise")
            except (TypeError, ValueError):
                return pd.NaT

        df[col] = df[col].apply(parse)
        return df

    def clean_priority(self, df, col):
        df[col] = df[col].astype(str).str.lower().map(self.priority_map)
        return df

    def clean_text(self, df, col):
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        return df

    def clean(self, df):
        df = df.copy()

        # 1) Nettoyer les dates
        df = self.clean_dates(df, "creation_date")

        # 2) Normaliser la priorité
        df = self.clean_priority(df, "priority")

        # 3) Nettoyer la description
        df = self.clean_text(df, "description")

        # 4) Supprimer les doublons
        df = df.drop_duplicates()

        # 5) Supprimer les lignes inutilisables
        df = df.dropna(subset=["client", "priority", "description"])
        return df


In [60]:
cleaner = ingest_csv_excel()

In [69]:
df_cleaned = cleaner.clean(df)
df_cleaned

,ticket_id,client,priority,creation_date,description
0,301.0,Stark Industries,Haute,2024-01-01,BUG: dashboard cannot load properly
2,302.0,Stark Industries,Basse,2024-01-02,Duplicate line from line 2
3,303.0,Oscorp,Critique,NaT,


# Convertion du tableau propre en document RAG

## Transformation du DataFrame propore en document RAG

In [67]:
from langchain_core.documents import Document
docs_final = []

for index, row in df_cleaned.iterrows():
    text = row_to_text(row)
    metadata = {"source": "support_dirty", "row_index": index}
    doc = Document(page_content=text, metadata=metadata)
    docs_final.append(doc)

print(f"Nombre de documents créés à partir du DataFrame nettoyé : {len(docs_final)}")

print("contenu d'un document du DataFrame nettoyé :")
print(docs_final[0].page_content)


Nombre de documents créés à partir du DataFrame nettoyé : 3
contenu d'un document du DataFrame nettoyé :
ticket_id: 301.0
client: Stark Industries
priority: Haute
creation_date: 2024-01-01 00:00:00
description: BUG: dashboard cannot load properly


In [73]:
# chunker les documents nettoyés
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)
final_chunks = splitter.split_documents(docs_final)

print(f"Nombre de chunks créés à partir des documents nettoyés : {len(final_chunks)}")
print("contenu d'un chunk du DataFrame nettoyé :")
print(final_chunks[0].page_content)

Nombre de chunks créés à partir des documents nettoyés : 3
contenu d'un chunk du DataFrame nettoyé :
ticket_id: 301.0
client: Stark Industries
priority: Haute
creation_date: 2024-01-01 00:00:00
description: BUG: dashboard cannot load properly
